In [1]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25  # install all required libraries for this project in one shot


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os  # gives us access to environment variables like API keys
from google.colab import userdata  # lets us read secrets stored in Colab's secret manager
from pageindex import PageIndexClient  # the client we use to talk to the PageIndex API

In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')  # pulls the Gemini key from Colab secrets and sets it as an env variable so LangChain picks it up automatically
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))  # creates an authenticated PageIndex client using our secret key


In [5]:
import json  # standard library module for reading/writing JSON files

with open("hackernews_corpus.json", "r", encoding="utf-8") as f:  # opens the scraped BBC corpus file in read mode, utf-8 handles special characters
    corpus = json.load(f)  # parses the JSON file into a Python list of article dicts

print("Number of articles:", len(corpus))  #confirms how many articles were loaded
print("\nFirst article title:")  # just a label before we print the title
print(corpus[0]["title"])  # prints the title of the first article to verify structure looks right


Number of articles: 30

First article title:
Antigravity 2.0 Tops the OpenSCAD Architectural 3D LLM Benchmark


In [6]:
!pip install spacy  # installs spaCy, used for NLP-based stopword removal and tokenisation
!python -m spacy download en_core_web_sm  # downloads the small English language model needed for spaCy to work

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
import re  # regex library for pattern-based text substitution
import spacy  # NLP library we use for tokenisation and stopword removal

nlp = spacy.load("en_core_web_sm")  # loads the small English spaCy model into memory

def clean_text(text):  # defines a function that takes raw article text and returns a cleaned version
    text = text.lower()  # converts everything to lowercase so 'Cuba' and 'cuba' are treated the same
    text = re.sub(r"[^\w\s]", " ", text)  # strips out all punctuation, replacing each match with a space
    text = re.sub(r"\s+", " ", text)  # collapses multiple consecutive spaces into a single space

    doc = nlp(text)  # runs the cleaned text through spaCy to get a processed doc object with token info
    tokens = []  # empty list to collect the tokens we actually want to keep
    for token in doc:  # iterates over every token spaCy identified in the text
        if not token.is_stop and not token.is_space:  # skips stopwords like 'the', 'is' and any whitespace tokens
            tokens.append(token.text)  # keeps the token if it passed both filters
    return " ".join(tokens)  # joins all kept tokens back into a single cleaned string


In [8]:
sample = corpus[0]["title"]  # grabs the raw title of the first article to use as a test case

cleaned = clean_text(sample)  # runs the first article through our cleaning function

print("ORIGINAL\n")  # label for the original text block
print(sample[:500])  # prints first 500 chars of the raw text so we can see what we started with
print("\n\nCLEANED\n")  # label for the cleaned text block
print(cleaned[:500])  # prints first 500 chars after cleaning so we can compare the before and after


ORIGINAL

Antigravity 2.0 Tops the OpenSCAD Architectural 3D LLM Benchmark


CLEANED

antigravity 2 0 tops openscad architectural 3d llm benchmark


In [9]:
cleaned_corpus = []  # empty list that will hold all cleaned articles once the loop finishes
for article in corpus:  # loops through every article dict in the raw corpus
    cleaned_corpus.append(
        {
            "title": article["title"],  # keeps the original title as it is
            "url": clean_text(article["url"])  # applies our clean_text function to the article body
        }
    )

print("Articles cleaned:", len(cleaned_corpus))  # confirms the cleaned corpus has the same count as the original


Articles cleaned: 30


In [10]:
with open(
    "cleaned_hackernews_corpus.json",  # name of the output file we're saving the cleaned corpus to
    "w",  # write mode creates the file if it doesn't exist, overwrites if it does
    encoding="utf-8"  # utf-8 so special characters like accented letters don't get mangled
) as f:
    json.dump(
        cleaned_corpus,  # the list of cleaned article dicts we built in the previous cell
        f,
        ensure_ascii=False,  # allows non-ASCII characters to be written as is instead of being escaped
        indent=4  # pretty-prints the JSON with 4-space indentation so it's human readable
    )

print("File saved successfully.")  # confirms the file write completed without errors

File saved successfully.


In [11]:
!pip install pageindex  # re-installs pageindex just to make sure it's available in this session
from pageindex import PageIndexClient  # imports the client class again after the fresh install
import time  # we'll use time.sleep later to wait between polling requests while the doc indexes

In [12]:
pi_client = PageIndexClient(
    api_key=userdata.get("PAGEINDEX")  # re-initialises the PageIndex client with our secret key  needed again after the re-install above
)


In [13]:
!pip install reportlab  # installs reportlab, a library for generating PDF files from Python
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak  # SimpleDocTemplate builds the PDF, Paragraph formats text, PageBreak inserts page separators
from reportlab.lib.styles import getSampleStyleSheet  # gives us pre-built text styles like Heading2 and BodyText

pdf = SimpleDocTemplate("hacker_news.pdf")  # creates a PDF document object pointing to the output file path
styles = getSampleStyleSheet()  # loads the default style sheet so we can reference heading and body styles
elements = []  # list that will hold all the content blocks in order before we build the PDF
for article in cleaned_corpus:  # loops through every cleaned article to add it to the PDF
    elements.append(
        Paragraph(
            f"{article['title']}",  # wraps the title in bold HTML tags so it renders as bold in the PDF
            styles["Heading2"]  # applies the Heading2 style so the title is visually larger
        )
    )
    elements.append(
        Paragraph(
            article["url"],  # the cleaned body text of the article
            styles["BodyText"]  # applies normal body paragraph formatting
        )
    )
    elements.append(PageBreak())  # inserts a hard page break so each article starts on its own page
pdf.build(elements)  # renders all the elements into the actual PDF file on disk
print("PDF created")  # confirms the PDF was written successfully

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.5 MB/s eta 0:00:00
PDF created


In [14]:
response = pi_client.submit_document(
    "hacker_news.pdf"  # sends the local PDF file to PageIndex for hierarchical indexing
)
doc_id = response["doc_id"]  # pulls the unique document ID from the response we need this for all future API calls
print(doc_id)  # prints the doc ID so we can reference it manually if needed


pi-cmpwydqs002x001quvqbrr4xx


In [15]:
while True:  # keeps looping until we explicitly break out this is a polling loop
    info = pi_client.get_document(doc_id)  # fetches the current status of our submitted document from PageIndex
    status = info["status"]  # extracts just the status field from the response dict
    print(status)  # prints the status each iteration so we can see it progressing
    if status == "completed":  # once the indexing job is done, we stop waiting
        break
    time.sleep(10)  # waits 10 seconds before checking again so we don't hammer the API with requests

processing
completed


In [16]:
tree = pi_client.get_tree(doc_id)  # fetches the full hierarchical tree structure that PageIndex built for our document

print(tree.keys())  # prints the top-level keys of the response so we can understand its structure before traversing it


dict_keys(['doc_id', 'status', 'retrieval_ready', 'result', 'metadata'])


In [17]:
documents = []  # empty list to collect all the node dicts as we walk through the PageIndex tree

for node in tree["result"]:  # iterates over each top-level node in the tree these are the sections PageIndex identified
    documents.append(
        {
            "node_id": node["node_id"],  # unique ID for this node within the PageIndex tree
            "title": node["title"],  # the section heading that PageIndex assigned to this chunk

        }
    )

In [18]:
!pip install rank-bm25  # installs the rank-bm25 library which gives us the BM25Okapi implementation
tokenized_corpus = [
    doc["title"].split()  # splits each document's text into a list of words BM25 needs tokenised input not raw strings
    for doc in documents  # runs this for every node we extracted from the PageIndex tree
]


In [19]:
from rank_bm25 import BM25Okapi  # imports the BM25Okapi variant Okapi BM25 is the standard probabilistic ranking formula

bm25 = BM25Okapi(tokenized_corpus)  # builds the BM25 index by computing term frequencies and IDF weights across all documents



In [20]:
import numpy as np  # numpy for argsort which lets us rank the BM25 scores efficiently

def retrieve(query, k=3):  # standalone retrieval function takes a query string and returns the top k matching documents
    query_tokens = query.lower().split()  # lowercases and tokenises the query the same way we tokenised the corpus
    scores = bm25.get_scores(query_tokens)  # scores every document in the index against the query tokens
    top_indices = np.argsort(scores)[::-1][:k]  # argsort gives ascending order, [::-1] reverses it to descending, [:k] takes the top k
    return [documents[i] for i in top_indices]  # returns the actual document dicts for the top k indices
results = retrieve(
    "What is happening between the US and Cuba?"  # test query to verify the retrieval is working before we wire it into LangChain
)
for doc in results:  # loops through the returned documents
    print(doc["title"])  # prints each retrieved document's title to see which chunks were ranked highest


Mycorrhizal Fungi, Nature’s Key to Plant Survival and Success
Deciphering the Hashihara Castle Town Map
Antigravity 2.0 Tops the OpenSCAD Architectural 3D LLM Benchmark


In [21]:
!pip install langchain langchain-community  # ensures langchain and its community extensions are installed

from langchain_core.documents import Document  # LangChain's Document class that wraps text content with metadata
from langchain_core.retrievers import BaseRetriever  # base class we'll subclass to make our BM25 retriever compatible with LangChain
from pydantic import Field  # used to declare typed fields on our custom retriever class so Pydantic validates them


In [24]:
langchain_docs = []  # empty list to hold the LangChain Document objects we're about to create
for doc in documents:  # loops through every node dict we pulled from the PageIndex tree
    langchain_docs.append(
        Document(
            page_content=doc["title"],  # the chunk's text becomes the page_content field that LangChain retrieval chains read from
            metadata={
                "title": doc["title"],  # storing the section title in metadata so we can inspect which chunk was retrieved
                "node_id": doc["node_id"]  # storing the PageIndex node ID in case we need to trace back to the source
            }
        )
    )

In [25]:
class BM25Retriever(BaseRetriever):  # custom retriever class that extends LangChain's BaseRetriever interface
    bm25: BM25Okapi = Field(...)  # declares the BM25 index as a required Pydantic field the ... means it has no default
    documents: list = Field(...)  # declares the list of LangChain Documents as a required field
    def _get_relevant_documents(
        self,
        query: str  # the search query string passed in by the retrieval chain
    ):
        query_tokens = query.lower().split()  # lowercases and splits the query so it matches how the corpus was tokenised
        scores = self.bm25.get_scores(
            query_tokens  # scores every document in the index against the query
        )
        top_indices = np.argsort(scores)[::-1][:3]  # sorts scores descending and takes the top 3 document indices
        return [
            self.documents[i]  # returns the actual LangChain Document objects for those top indices
            for i in top_indices
        ]
retriever = BM25Retriever(
    bm25=bm25,  # passes in the BM25 index we built earlier
    documents=langchain_docs  # passes in the list of LangChain Document objects
)

In [26]:
results = retriever.invoke(
    "What charges were filed against Raul Castro?"  # test query sent through the LangChain-compatible retriever
)
for doc in results:  # loops through the top retrieved documents
    print(doc.metadata["title"])  # prints each retrieved chunk's title to verify the retriever is returning relevant results


Deciphering the Hashihara Castle Town Map
Launch HN: Runtime (YC P26) – Sandboxed coding agents for everyone on a team
Mycorrhizal Fungi, Nature’s Key to Plant Survival and Success


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI  # imports the LangChain wrapper for Google's Gemini models

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # uses the Gemini 2.5 Flash model fast and capable for Q&A tasks
    temperature=0  # sets temperature to 0 so the model gives deterministic, factual answers with no randomness
)

In [28]:
from langchain_core.prompts import ChatPromptTemplate  # imports the class for building structured chat prompts
prompt = ChatPromptTemplate.from_template(
    """
You are a question-answering assistant.
Answer ONLY using the provided context.
If the answer cannot be found in the context, say:
\"I could not find the answer in the provided documents.\"
Context:
{context}
Question:
{input}
Answer:
"""
)

In [29]:
import langchain  # imports the main langchain package
print(dir(langchain))  # prints all names exported by langchain used here to explore what's available after install


['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__']


In [30]:
import langchain_classic  # imports langchain_classic which contains the older-style chain builders we need
print("langchain_classic imported")  # confirms the import succeeded useful for debugging version issues


langchain_classic imported


In [31]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain  # imports the function that builds a chain to stuff retrieved docs into the prompt

document_chain = create_stuff_documents_chain(
    llm,  # the Gemini LLM that will generate the final answer
    prompt  # the prompt template we defined earlier that slots in context and question
)



In [32]:
from langchain_classic.chains import create_retrieval_chain  # imports the function that wraps a retriever and a document chain into a full RAG chain

retrieval_chain = create_retrieval_chain(
    retriever,  # our custom BM25Retriever that fetches relevant document chunks for a given query
    document_chain  # the chain that formats those chunks into the prompt and passes it to the LLM
)


In [39]:
response = retrieval_chain.invoke({
    "input": "Steve Wozniak's comments on AI?"  # the question we want answered the chain will retrieve relevant chunks and pass them to Gemini
})

print(response["answer"])  # prints the LLM's answer it should be grounded in the retrieved hackernews article chunks


I could not find the answer in the provided documents.


In [40]:
full_corpus = "\n\n".join(
    [doc.page_content for doc in langchain_docs]  # concatenates every document chunk's text into one giant string separated by double newlines
)  # this becomes the naive approach's context the entire corpus dumped in at once instead of retrieving relevant bits


In [44]:
start = time.time()  # records the start time before we call the LLM
response = llm.invoke(
    f"""
Context:
{full_corpus}
Question:
Steve Wozniak's comments on AI?
"""
)
naive_time = time.time() - start  # calculates total time taken for the naive full-context approach
print(response.content)  # prints the LLM's answer using the full corpus as context
print("Time:", naive_time)  # prints how many seconds the naive approach took expected to be slower than RAG


Steve Wozniak told students they have AI, which he defined as "actual intelligence."
Time: 1.8827922344207764


In [45]:
start = time.time()  # records the start time before invoking the RAG chain
response = retrieval_chain.invoke({
    "input": "Steve Wozniak's comments on AI?"  # same question as the naive test so results are directly comparable
})
rag_time = time.time() - start  # calculates how long the full RAG pipeline (retrieval + LLM) took
print(response["answer"])  # prints the answer generated by the BM25 RAG chain
print("Time:", rag_time)  # prints the RAG latency should be faster than naive since we only send 3 chunks not the full corpus

I could not find the answer in the provided documents.
Time: 2.138458013534546
